# Epydemix Exercise: Modeling an Epidemic with Population Data and Interventions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ngozzi/computational-epidemiology/blob/main/notebooks/epydemix/epydemix_exercise.ipynb)

In this exercise, you will apply the concepts from Tutorials 1–3 to model a fictional respiratory disease called **Virus-X** spreading through a real-world population. You will:

1. **Define and simulate** an SEIR model from scratch using epydemix
2. **Load real population data** and run simulations for epidemic dynamics across countries
3. **Model interventions** and quantify their effectiveness

If you are running this on Google Colab, install the required packages first:

In [ ]:
import sys, os, subprocess
if "google.colab" in sys.modules or os.getenv("COLAB_RELEASE_TAG"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "https://raw.githubusercontent.com/epistorm/epydemix/refs/heads/main/tutorials/colab_requirements.txt"])

## Part 1: Defining and Simulating an SEIR Model

**Virus-X** has an incubation period before individuals become infectious. This means the standard SIR model is insufficient — we need an **SEIR model**, which adds an **Exposed (E)** compartment:

$$S \xrightarrow{\beta \cdot I} E \xrightarrow{\sigma} I \xrightarrow{\gamma} R$$

The transitions are:
- **S → E**: Mediated by I, at rate $\beta$ (transmission rate)
- **E → I**: Spontaneous, at rate $\sigma$ (incubation rate, i.e. $1/\sigma$ is the average incubation period)
- **I → R**: Spontaneous, at rate $\gamma$ (recovery rate)

**Disease parameters for Virus-X:**
- Transmission rate: $\beta = 0.35$
- Incubation rate: $\sigma = 1/5$ (5-day incubation period)
- Recovery rate: $\gamma = 1/7$ (7-day infectious period)

**Task 1.1**: Define the SEIR model using `EpiModel`. Add the compartments and the three transitions listed above.

In [ ]:
from epydemix import EpiModel

## complete the code below
seir_model = EpiModel(
    name='SEIR Model - Virus-X',
    compartments=None,  # TODO: list the four compartments: S, E, I, R
)

# TODO: add the three transitions
# seir_model.add_transition(...)
# seir_model.add_transition(...)
# seir_model.add_transition(...)

print(seir_model)

**Task 1.2**: Run 100 stochastic simulations from `2024-01-01` to `2024-06-30` with 20 initially infected individuals (use `percentage_in_agents` accordingly; the default population size is 100,000).

In [ ]:
## complete the code below
seir_results = seir_model.run_simulations(
    start_date=None,       # TODO
    end_date=None,         # TODO
    Nsim=None,             # TODO
    percentage_in_agents=None  # TODO: 20 / total population
)

**Task 1.3**: Plot the quantiles of all four compartments (S, E, I, R) over time.

In [ ]:
from epydemix.visualization import plot_quantiles

## complete the code below
df_quantiles = seir_results.get_quantiles_compartments()

# TODO: call plot_quantiles with S_total, E_total, I_total, R_total
ax = plot_quantiles(df_quantiles, columns=None, title='SEIR Model - Virus-X (default population)')

**Task 1.4**: Plot the transition quantiles for `S_to_E_total` (new exposures) over time. This represents the daily incidence of the disease.

In [ ]:
## complete the code below
df_transitions = seir_results.get_quantiles_transitions()

# TODO: call plot_quantiles with S_to_E_total
ax = plot_quantiles(df_transitions, columns=None, title='Daily new exposures - Virus-X')

---

## Part 2: Modeling with Real Population Data

Virus-X is spreading globally. You are tasked with running simulations for its impact in **Italy** or other countries with very different population age structures and contact patterns.

**Task 2.1**: Load the population objects for Italy using `load_epydemix_population`.

In [ ]:
from epydemix.population import load_epydemix_population

## complete the code below
population_italy = None  # TODO
print(population_italy)

**Task 2.2**: Visualize the contact matrices for all four layers (school, work, home, community) for **Italy** using `plot_contact_matrix`. Arrange them in a 2×2 grid of subplots.

In [ ]:
from epydemix.visualization import plot_contact_matrix
import matplotlib.pyplot as plt

## complete the code below
fig, axes = plt.subplots(nrows=2, ncols=2, dpi=200, figsize=(10, 8))

# TODO: plot contact matrices for school, work, home, community (Italy)

plt.suptitle("Italy — Contact Matrices", fontsize=14)
plt.tight_layout()

**Task 2.3**: Create an SEIR model for each country (same Virus-X parameters as in Part 1), set the respective population, and run 100 simulations for each country from `2024-01-01` to `2024-09-30`. Use 100 initially infected individuals.

In [ ]:
from epydemix import load_predefined_model

## complete the code below

# --- Italy ---
model_italy = load_predefined_model("SEIR", transmission_rate=0.04, recovery_rate=1/7, incubation_rate=1/5)
# TODO: set population for Italy

results_italy = model_italy.run_simulations(
    start_date=None,  # TODO
    end_date=None,    # TODO
    Nsim=None,        # TODO
    percentage_in_agents=None  # TODO: 100 / Italy population
)

**Task 2.4**: Plot the infected compartment (`Infected_total`). Use `plot_quantiles` with different colors.

In [ ]:
import seaborn as sns
colors = sns.color_palette("Dark2")

## complete the code below
fig, ax = plt.subplots(figsize=(10, 5), dpi=200)

df_italy = results_italy.get_quantiles_compartments()

# TODO: plot infected compartment

ax.set_title("Virus-X: Fraction Infected over Time")
plt.show()

In [ ]:
import numpy as np

## complete the code below

# get full trajectories
traj_italy = results_italy.get_stacked_compartments()
traj_kenya = results_kenya.get_stacked_compartments()

# TODO: compute attack rate for each simulation run
ar_italy = None  # shape: (Nsim,)
ar_kenya = None  # shape: (Nsim,)

# TODO: print median and 95% CI for each country
print("Italy  — Attack rate: ...")
print("Kenya  — Attack rate: ...")

---

## Part 3: Modeling Non-Pharmaceutical Interventions

Health authorities in Italy decide to implement a series of interventions to slow the spread of Virus-X. You are asked to model two scenarios:

- **Scenario A – No interventions**: the baseline model from Part 2 (Italy, no interventions)
- **Scenario B – With interventions**: the same model, but with the following measures:

  1. **School closure** (`school` layer): from `2024-02-01` to `2024-03-15`, contacts in schools are reduced by 80% (reduction factor: 0.2)
  2. **Workplace reduction** (`work` layer): from `2024-02-15` to `2024-04-30`, contacts at work are reduced by 50% (reduction factor: 0.5)
  3. **Reduced transmission** (masking + distancing): from `2024-02-01` to `2024-09-30`, override `transmission_rate` to `0.025`

**Task 3.1**: Create a new SEIR model for Italy **with interventions**. Add all three interventions described above.

In [ ]:
## complete the code below
model_italy_npi = load_predefined_model("SEIR", transmission_rate=0.04, recovery_rate=1/7, incubation_rate=1/5)
model_italy_npi.set_population(population_italy)

# TODO: add school closure intervention
# model_italy_npi.add_intervention(...)

# TODO: add workplace reduction intervention
# model_italy_npi.add_intervention(...)

# TODO: override transmission_rate during the intervention period
# model_italy_npi.override_parameter(...)

**Task 3.2**: Visualize the impact of the interventions on contact dynamics using `plot_spectral_radius`. This shows how the overall transmission potential changes over time.

In [ ]:
from epydemix.visualization import plot_spectral_radius
from epydemix.utils import compute_simulation_dates

simulation_dates = compute_simulation_dates("2024-01-01", "2024-09-30")

## complete the code below
# TODO: call compute_contact_reductions and then plot_spectral_radius
# model_italy_npi.compute_contact_reductions(simulation_dates)
# plot_spectral_radius(...)

**Task 3.3**: Run 100 simulations for the intervention scenario (same dates and initial conditions as for Italy in Part 2).

In [ ]:
## complete the code below
results_italy_npi = model_italy_npi.run_simulations(
    start_date=None,  # TODO
    end_date=None,    # TODO
    Nsim=None,        # TODO
    percentage_in_agents=None  # TODO: 100 / Italy population
)

**Task 3.4**: Compare the two Italy scenarios (with vs. without interventions) by plotting the `Infected_total` compartment for both on the same axes.

In [ ]:
## complete the code below
fig, ax = plt.subplots(figsize=(10, 5), dpi=200)

# TODO: plot both scenarios
# Hint: use the ax argument of plot_quantiles to overlay on the same axes

ax.set_title("Italy — Virus-X: With vs. Without Interventions")
plt.show()

**Task 3.5**: Quantify the effect of interventions by computing:
1. **Averted infections** (% reduction in total infections)
2. **Peak size reduction** (% reduction in peak infected)

Report the median and 95% CI across simulations using a boxplot.

In [ ]:
## complete the code below
traj_italy_npi = results_italy_npi.get_stacked_compartments()
traj_italy_baseline = results_italy.get_stacked_compartments()

# TODO: compute averted infections (%)
averted_infections_perc = None

# TODO: compute peak size reduction (%)
delta_peak_perc = None

# TODO: create a 1x2 boxplot figure
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4), dpi=200)

# ...

axes[0].set_title("Averted infections (%)")
axes[1].set_title("Peak size reduction (%)")
plt.tight_layout()
plt.show()